# S04 — NumPy: Arrays, Broadcasting & Vectorised Physics

**Python for Applied Physics** | Master in Applied Physics  
**Prerequisites:** S01 (Python basics), S02 (functions), S03 (project structure & Git)

---

## Learning objectives

By the end of this session you will be able to:

- Create, inspect, and manipulate NumPy arrays efficiently
- Use slicing, boolean masking, and fancy indexing to extract data
- Write vectorised code that replaces explicit loops
- Apply broadcasting to compute 2D physical fields from 1D grids
- Perform basic linear algebra for optical system simulation
- Compute pulse spectra with the FFT and interpret time–bandwidth products
- Extend `shared/optics.py` and `shared/pulses.py` with NumPy-based functions

---

## Session map

| Block | Topic | Cells |
|-------|-------|-------|
| 1 | The NumPy array | 1–7 |
| 2 | Indexing & slicing | 8–14 |
| 3 | Vectorised operations & ufuncs | 15–21 |
| 4 | Broadcasting | 22–28 |
| 5 | Linear algebra | 29–33 |
| 6 | FFT | 34–39 |
| 7 | Practical tips & I/O | 40–43 |

In [2]:
import numpy as np
import sys

print(f"Python {sys.version}")
print(f"NumPy  {np.__version__}")

Python 3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ]
NumPy  2.4.3


---
## Block 1 — The NumPy array

A Python list stores a collection of *pointers* to arbitrary objects.  
A NumPy `ndarray` stores a *contiguous block of typed memory* — like a C array, but with a Python interface.

This is why NumPy is fast: one contiguous block → cache-friendly → SIMD vectorisation by the CPU.

```
Python list:   [ * | * | * | * ]   each * → arbitrary object (int, str, …)
NumPy array:   [ f64 | f64 | f64 | f64 ]   raw 8-byte floats, no boxing
```

In [2]:
# --------------------------------------------------------------------------
# Array creation
# --------------------------------------------------------------------------

# From a Python list
wavelengths = np.array([800e-9, 1030e-9, 1550e-9])   # m
print(f"wavelengths: {wavelengths}")
print(f"dtype : {wavelengths.dtype}")

# Integer dtype inferred automatically
counts = np.array([1, 2, 3, 4])
print(f"counts dtype: {counts.dtype}")

# Force dtype
counts_f = np.array([1, 2, 3, 4], dtype=np.float64)
print(f"counts_f dtype: {counts_f.dtype}")

wavelengths: [8.00e-07 1.03e-06 1.55e-06]
dtype : float64
counts dtype: int64
counts_f dtype: float64


In [3]:
# --------------------------------------------------------------------------
# Standard constructors
# --------------------------------------------------------------------------

z = np.zeros(5)
o = np.ones((3, 3))
eye = np.eye(4)                        # identity matrix
full = np.full((2, 4), fill_value=np.pi)

# For grids: linspace is almost always preferable to arange with floats
t = np.linspace(-1e-12, 1e-12, 1024)  # 1024 points from -1 ps to +1 ps
n = np.arange(0, 10, 2)               # [0 2 4 6 8]  — integer steps only

print(f"t: {t.shape}, first={t[0]:.3e}, last={t[-1]:.3e}")
print(f"n: {n}")

t: (1024,), first=-1.000e-12, last=1.000e-12
n: [0 2 4 6 8]


💡 **Pro Tip — `linspace` vs `arange` for floats**

```python
# WRONG: floating-point rounding can give 11 or 9 points
t = np.arange(0.0, 1.0, 0.1)   # ← how many elements?

# RIGHT: always gives exactly N points, endpoints guaranteed
t = np.linspace(0.0, 1.0, 11)
```

Rule of thumb: use `arange` only when the step is an integer or you specifically need a *step size* rather than a *number of points*.

In [4]:
# --------------------------------------------------------------------------
# Anatomy of an ndarray
# --------------------------------------------------------------------------

a = np.ones((4, 3, 2))   # 3D array

print(f"shape    : {a.shape}")
print(f"ndim     : {a.ndim}")
print(f"size     : {a.size}   (total elements)")
print(f"dtype    : {a.dtype}")
print(f"itemsize : {a.itemsize} bytes per element")
print(f"nbytes   : {a.nbytes} bytes total")

shape    : (4, 3, 2)
ndim     : 3
size     : 24   (total elements)
dtype    : float64
itemsize : 8 bytes per element
nbytes   : 192 bytes total


In [5]:
# --------------------------------------------------------------------------
# Reshaping — no data is copied, only the shape metadata changes
# --------------------------------------------------------------------------

v = np.arange(12)
M = v.reshape(3, 4)     # 3 rows × 4 cols
col = v.reshape(-1, 1)  # -1 = infer dimension  →  column vector (12, 1)

print(f"v : {v.shape}")
print(f"M : {M.shape}")
print(f"col : {col.shape}")

v : (12,)
M : (3, 4)
col : (12, 1)


⚠️ **Common Pitfall — copy vs view**

```python
a = np.array([1.0, 2.0, 3.0])
b = a          # b is just another name for the same array
b[0] = 99.0
print(a[0])    # 99.0  ← a was modified!

# Fix: explicit copy
b = a.copy()
b[0] = 99.0
print(a[0])    # 1.0  ← a is safe
```

Assignment (`b = a`) never copies data — it creates a second reference to the same object.  
This is true for all Python mutable objects, not just NumPy arrays.

In [6]:
# Demonstrate the pitfall
a = np.array([1.0, 2.0, 3.0])
b = a           # view (alias)
c = a.copy()    # independent copy

b[0] = 99.0
print(f"a[0] after modifying b : {a[0]}")  # 99 — same data
print(f"c[0] after modifying b : {c[0]}")  # 1.0 — copy is safe

# Check with np.shares_memory
print(f"a and b share memory: {np.shares_memory(a, b)}")
print(f"a and c share memory: {np.shares_memory(a, c)}")

a[0] after modifying b : 99.0
c[0] after modifying b : 1.0
a and b share memory: True
a and c share memory: False


---
## Block 2 — Indexing & slicing

NumPy extends Python's slice syntax to multiple dimensions.  
The general syntax is `a[start:stop:step, start:stop:step, ...]`.

In [7]:
# --------------------------------------------------------------------------
# 1D slicing — identical to Python lists
# --------------------------------------------------------------------------

t = np.linspace(0, 10e-12, 11)   # 0 to 10 ps, 11 points
print(f"t      : {t * 1e12} ps")
print(f"t[2:5] : {t[2:5] * 1e12} ps")
print(f"t[::2] : {t[::2] * 1e12} ps")   # every other point
print(f"t[::-1]: {t[::-1] * 1e12} ps")  # reversed

t      : [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.] ps
t[2:5] : [2. 3. 4.] ps
t[::2] : [ 0.  2.  4.  6.  8. 10.] ps
t[::-1]: [10.  9.  8.  7.  6.  5.  4.  3.  2.  1.  0.] ps


In [8]:
# --------------------------------------------------------------------------
# 2D slicing — rows and columns separated by comma
# --------------------------------------------------------------------------

# Simulate a beam image: 5×5 fluence grid (J/cm²)
F = np.array([
    [0.1, 0.3, 0.5, 0.3, 0.1],
    [0.3, 0.7, 1.0, 0.7, 0.3],
    [0.5, 1.0, 2.0, 1.0, 0.5],
    [0.3, 0.7, 1.0, 0.7, 0.3],
    [0.1, 0.3, 0.5, 0.3, 0.1],
])

print("Central row (row 2):")
print(F[2, :])         # all columns, row 2

print("\nCentral column (col 2):")
print(F[:, 2])         # all rows, col 2

print("\nCentral 3×3 sub-array:")
print(F[1:4, 1:4])

Central row (row 2):
[0.5 1.  2.  1.  0.5]

Central column (col 2):
[0.5 1.  2.  1.  0.5]

Central 3×3 sub-array:
[[0.7 1.  0.7]
 [1.  2.  1. ]
 [0.7 1.  0.7]]


In [9]:
# --------------------------------------------------------------------------
# Boolean masking
# --------------------------------------------------------------------------

# Extract pixels above a damage threshold
threshold = 1.5   # J/cm²
mask = F > threshold
print("Damage mask:")
print(mask.astype(int))

print(f"\nPixels above threshold: {F[mask]}")   # 1D array of surviving values
print(f"Number of damaged pixels: {mask.sum()}")

# Set damaged pixels to zero (in-place)
F_clipped = F.copy()
F_clipped[mask] = threshold
print(f"\nMax after clipping: {F_clipped.max():.1f} J/cm²")

Damage mask:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Pixels above threshold: [2.]
Number of damaged pixels: 1

Max after clipping: 1.5 J/cm²


In [ ]:
# --------------------------------------------------------------------------
# Fancy indexing — indexing with an array of indices
# --------------------------------------------------------------------------

wavelengths = np.array([400, 532, 800, 1030, 1550])   # nm
energies    = np.array([10,  25,  50,   30,   15])    # µJ

# Select channels by index array
vis_idx = np.array([0, 1])          # 400 and 532 nm
print(f"Visible wavelengths: {wavelengths[vis_idx]} nm")
print(f"Visible energies   : {energies[vis_idx]} µJ")

# Sort by energy (fancy indexing with argsort)
order = np.argsort(energies)[::-1]  # descending
print(f"\nChannels sorted by energy (desc):")
for wl, en in zip(wavelengths[order], energies[order]):
    print(f"  {wl} nm → {en} µJ")

⚠️ **Common Pitfall — slices return views, fancy indexing returns copies**

```python
a = np.array([1.0, 2.0, 3.0, 4.0])

# Slice → VIEW (modifying s modifies a)
s = a[1:3]
s[0] = 99.0
print(a)   # [1. 99. 3. 4.]

# Fancy indexing → COPY (safe)
f = a[[1, 2]]
f[0] = 0.0
print(a)   # [1. 99. 3. 4.]  ← unchanged
```

When in doubt, use `np.shares_memory(a, s)` to check.

In [10]:
# Verify: slice is a view, fancy index is a copy
a = np.array([1.0, 2.0, 3.0, 4.0])

s = a[1:3]        # slice
f = a[[1, 2]]     # fancy

print(f"slice shares memory : {np.shares_memory(a, s)}")
print(f"fancy shares memory : {np.shares_memory(a, f)}")

slice shares memory : True
fancy shares memory : False


---
## Block 3 — Vectorised operations & ufuncs

A **ufunc** (universal function) applies an operation element-wise to an array with no Python loop.  
The loop runs in compiled C — orders of magnitude faster than a Python `for` loop.

In [ ]:
# --------------------------------------------------------------------------
# Elementwise arithmetic
# --------------------------------------------------------------------------

λ  = np.array([800e-9, 1030e-9, 1550e-9])   # m
c  = 3e8                                     # m/s
ν  = c / λ                                   # Hz — vectorised division

print("Frequencies (THz):")
for wl, freq in zip(λ * 1e9, ν / 1e12):
    print(f"  λ={wl:.0f} nm → ν={freq:.1f} THz")

In [ ]:
# --------------------------------------------------------------------------
# 🔬 Physics Insight — Gaussian beam intensity profile
#
#   I(r) = I₀ · exp(-2r²/w²)
#
# where w is the 1/e² beam radius and I₀ = 2P/(πw²) is the peak intensity
# --------------------------------------------------------------------------

# Beam parameters
w     = 500e-6    # 500 µm beam radius
P     = 1.0       # 1 W average power
I0    = 2 * P / (np.pi * w**2)   # peak intensity (W/m²)

# Radial grid: 512 points from -3w to +3w
r = np.linspace(-3 * w, 3 * w, 512)

# Vectorised — no loop needed
I = I0 * np.exp(-2 * r**2 / w**2)

print(f"Peak intensity : {I0 / 1e4:.1f} W/cm²")
print(f"I at r=w      : {I[256 + int(512 * w / (6*w))] / 1e4:.2f} W/cm²")
print(f"I(w)/I₀       : {np.exp(-2):.4f}  (expected {np.exp(-2):.4f})")

In [ ]:
# --------------------------------------------------------------------------
# Common ufuncs
# --------------------------------------------------------------------------

x = np.linspace(-np.pi, np.pi, 5)
print(f"x            : {x}")
print(f"np.sin(x)    : {np.sin(x).round(4)}")
print(f"np.exp(x)    : {np.exp(x).round(4)}")
print(f"np.abs(x)    : {np.abs(x).round(4)}")

# Complex arrays — np.angle extracts the phase
z = np.exp(1j * x)
print(f"\nnp.angle(z)  : {np.angle(z).round(4)}")   # same as x here
print(f"np.abs(z)    : {np.abs(z).round(4)}")       # 1 for unit complex

In [ ]:
# --------------------------------------------------------------------------
# Aggregations along axes + cumsum / diff
# --------------------------------------------------------------------------

# Simulate energy readings (µJ) across 4 laser shots × 3 channels
E = np.array([
    [10.1, 25.3, 50.2],
    [10.0, 24.9, 49.8],
    [10.2, 25.1, 50.5],
    [9.9,  25.0, 50.0],
])

print(f"Per-channel sum  : {E.sum(axis=0)}")    # sum over shots (axis 0)
print(f"Per-shot total   : {E.sum(axis=1)}")    # sum over channels (axis 1)
print(f"Per-channel max  : {E.max(axis=0)}")
print(f"Global argmax    : {np.unravel_index(E.argmax(), E.shape)}")  # (row, col)

# cumsum — running total of deposited energy
cumulative = np.cumsum(E[:, 2])   # cumulative energy, channel 2
print(f"\nCumulative energy ch2 (µJ): {cumulative}")

# diff — shot-to-shot variation
delta = np.diff(E[:, 2])
print(f"Shot-to-shot δE ch2 (µJ) : {delta}")

🔬 **Physics Insight — `np.diff` and group delay**

In ultrafast optics the group delay is defined as:

$$\tau_g(\omega) = -\frac{d\phi}{d\omega}$$

Numerically: `group_delay = -np.diff(phase) / np.diff(omega)`. You'll use this in S06 when we work with spectral phase data.

In [ ]:
# ⚡ Try It — vectorised speed comparison
#
# Compute I(r) = I0 * exp(-2r²/w²) for 10⁶ points
# (a) with a Python loop  (b) vectorised
# Compare execution times with %timeit

import time

N  = 1_000_000
r  = np.linspace(-3 * w, 3 * w, N)

# (a) Python loop
t0 = time.perf_counter()
I_loop = [I0 * np.exp(-2 * ri**2 / w**2) for ri in r]
t_loop = time.perf_counter() - t0

# (b) Vectorised
t0 = time.perf_counter()
I_vec = I0 * np.exp(-2 * r**2 / w**2)
t_vec = time.perf_counter() - t0

print(f"Loop      : {t_loop*1e3:.1f} ms")
print(f"Vectorised: {t_vec*1e3:.1f} ms")
print(f"Speedup   : {t_loop/t_vec:.0f}×")

---
## Block 4 — Broadcasting

Broadcasting allows NumPy to operate on arrays of *different shapes* without copying data.

**Rules (applied from the trailing dimension):**

1. If arrays have different numbers of dimensions, prepend 1s to the smaller shape.
2. Dimensions of size 1 are *stretched* to match the other array.
3. If any non-1 dimensions disagree → error.

```
Shape (512,)   →  (1, 512)  →  (512, 512)
Shape (512, 1) →  (512, 1)  →  (512, 512)
                              compatible ✓
```

In [3]:
# --------------------------------------------------------------------------
# Broadcasting fundamentals
# --------------------------------------------------------------------------

# Scalar broadcast
a = np.array([1.0, 2.0, 3.0])
print(f"a * 10 : {a * 10}")   # scalar stretches to match (3,)

# 1D × 1D — both must have size 1 or match
row = np.array([[1, 2, 3]])    # shape (1, 3)
col = np.array([[10], [20]])   # shape (2, 1)
print(f"\nrow shape: {row.shape}")
print(f"col shape: {col.shape}")
print(f"row + col:\n{row + col}")   # (2, 3) — outer sum

a * 10 : [10. 20. 30.]

row shape: (1, 3)
col shape: (2, 1)
row + col:
[[11 12 13]
 [21 22 23]]


In [ ]:
# --------------------------------------------------------------------------
# np.newaxis — explicit dimension insertion
# --------------------------------------------------------------------------

x = np.linspace(-1, 1, 5)   # shape (5,)

# Make column vector and row vector for outer operations
x_col = x[:, np.newaxis]    # shape (5, 1)
x_row = x[np.newaxis, :]    # shape (1, 5)

print(f"x_col shape : {x_col.shape}")
print(f"x_row shape : {x_row.shape}")

outer_sum = x_col + x_row   # (5, 5)
print(f"outer_sum shape : {outer_sum.shape}")

In [ ]:
# --------------------------------------------------------------------------
# 🔬 Physics Insight — 2D Gaussian field via broadcasting
#
#   E(x, y) = E₀ · exp(-(x² + y²) / w²)
#
# We build a 2D field from two 1D coordinate arrays WITHOUT a nested loop.
# --------------------------------------------------------------------------

N  = 256
w  = 500e-6   # 500 µm
E0 = 1.0

# 1D coordinate grids
x = np.linspace(-3 * w, 3 * w, N)    # shape (N,)
y = np.linspace(-3 * w, 3 * w, N)    # shape (N,)

# Promote to 2D via broadcasting
X = x[np.newaxis, :]   # shape (1, N)
Y = y[:, np.newaxis]   # shape (N, 1)

# Both broadcast to (N, N) — no loop, no meshgrid copy
E_field = E0 * np.exp(-(X**2 + Y**2) / w**2)   # shape (N, N)

print(f"E_field shape  : {E_field.shape}")
print(f"Peak at centre : {E_field[N//2, N//2]:.4f}")
print(f"Memory         : {E_field.nbytes / 1024:.1f} kB")

In [ ]:
# --------------------------------------------------------------------------
# np.meshgrid — explicit 2D grids (equivalent but uses more memory)
# --------------------------------------------------------------------------

X_mg, Y_mg = np.meshgrid(x, y)   # each shape (N, N)
E_mg = E0 * np.exp(-(X_mg**2 + Y_mg**2) / w**2)

print(f"meshgrid result matches broadcasting: {np.allclose(E_field, E_mg)}")
print(f"meshgrid memory: {X_mg.nbytes + Y_mg.nbytes} bytes for the grid arrays alone")

---
## Block 5 — Linear algebra

NumPy's `linalg` module covers the linear algebra operations you need for optical system modelling.

In [ ]:
# --------------------------------------------------------------------------
# Matrix multiplication: @ operator vs np.dot
# --------------------------------------------------------------------------

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A @ B:")
print(A @ B)          # preferred: the @ operator (PEP 465, Python 3.5+)

print("\nnp.dot(A, B):")
print(np.dot(A, B))   # equivalent for 2D arrays

# Element-wise multiplication (NOT matrix multiplication)
print("\nA * B (element-wise):")
print(A * B)

In [ ]:
# --------------------------------------------------------------------------
# 🔬 Physics Insight — ABCD ray transfer matrices
#
# A ray is represented as [y, θ] (height, angle).
# Propagation through an optical element is a 2×2 matrix multiplication.
# --------------------------------------------------------------------------

def free_space(d: float) -> np.ndarray:
    """ABCD matrix for free-space propagation over distance d."""
    return np.array([[1, d], [0, 1]])

def thin_lens(f: float) -> np.ndarray:
    """ABCD matrix for a thin lens with focal length f."""
    return np.array([[1, 0], [-1/f, 1]])

# System: propagate 500 mm → lens f=250 mm → propagate 500 mm
M_total = free_space(0.5) @ thin_lens(0.25) @ free_space(0.5)

print("System ABCD matrix:")
print(M_total.round(6))

# A collimated ray (θ=0) at height y=5 mm
ray_in  = np.array([5e-3, 0.0])   # [y, θ]
ray_out = M_total @ ray_in
print(f"\nInput  : y={ray_in[0]*1e3:.1f} mm, θ={ray_in[1]*1e3:.1f} mrad")
print(f"Output : y={ray_out[0]*1e6:.2f} µm, θ={ray_out[1]*1e3:.2f} mrad")

In [ ]:
# --------------------------------------------------------------------------
# 🔬 Physics Insight — Jones calculus for polarisation
#
# A polarisation state is a 2-component complex vector [Ex, Ey].
# Optical elements are 2×2 complex matrices.
# --------------------------------------------------------------------------

def linear_polariser(angle_deg: float) -> np.ndarray:
    """Jones matrix for a linear polariser at angle θ from x-axis."""
    θ = np.radians(angle_deg)
    c, s = np.cos(θ), np.sin(θ)
    return np.array([[c**2, c*s], [c*s, s**2]])

def half_wave_plate(fast_axis_deg: float) -> np.ndarray:
    """Jones matrix for a half-wave plate with fast axis at angle θ."""
    θ = np.radians(fast_axis_deg)
    c, s = np.cos(2*θ), np.sin(2*θ)
    return np.array([[c, s], [s, -c]])

# Input: horizontal linear polarisation
E_in = np.array([1.0 + 0j, 0.0 + 0j])

# Rotate to 45° with a HWP (fast axis at 22.5°)
E_out = half_wave_plate(22.5) @ E_in
print(f"After HWP(22.5°): Ex={E_out[0].real:.3f}, Ey={E_out[1].real:.3f}")
print(f"Intensity preserved: {np.abs(E_out)**2}")

# Pass through a vertical analyser
E_final = linear_polariser(90) @ E_out
T = np.sum(np.abs(E_final)**2)
print(f"Transmission through vertical analyser: {T:.3f}")

In [ ]:
# --------------------------------------------------------------------------
# np.linalg — eigenvalues, inverse, solve
# --------------------------------------------------------------------------

# Eigenvalues of the ABCD round-trip matrix (stability analysis)
M_rt = M_total @ M_total   # double pass (round trip)
eigenvalues, eigenvectors = np.linalg.eig(M_rt)
print(f"Round-trip eigenvalues: {eigenvalues}")

# Solve linear system  Ax = b
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([5.0, 10.0])
x = np.linalg.solve(A, b)   # preferred over np.linalg.inv(A) @ b
print(f"\nSolution x: {x}")
print(f"Verification A @ x: {A @ x}")   # should equal b

---
## Block 6 — FFT

The Discrete Fourier Transform (DFT) computed by `np.fft.fft` maps a time-domain signal to its frequency-domain representation.

For a Gaussian pulse:

$$E(t) = E_0 \exp\!\left(-\frac{t^2}{2\tau^2}\right) e^{i\omega_0 t}$$

the spectrum is also Gaussian. The time–bandwidth product for a transform-limited pulse is:

$$\Delta t \cdot \Delta\nu \geq \frac{2\ln 2}{\pi} \approx 0.4413 \quad (\text{for Gaussian pulses, FWHM})$$

In [ ]:
# --------------------------------------------------------------------------
# FFT of a Gaussian pulse
# --------------------------------------------------------------------------

# Pulse parameters
τ    = 100e-15    # 100 fs pulse duration (1/e field half-width)
λ0   = 800e-9    # 800 nm central wavelength
c    = 3e8
ω0   = 2 * np.pi * c / λ0   # central angular frequency

# Time grid — must be fine enough and long enough
N  = 4096
dt = 10e-15      # 10 fs sampling interval
t  = (np.arange(N) - N // 2) * dt   # centred on zero

# Electric field envelope (real-valued Gaussian)
E_t = np.exp(-t**2 / (2 * τ**2))   # amplitude envelope

print(f"Time window : {t[0]*1e12:.1f} ps to {t[-1]*1e12:.1f} ps")
print(f"dt          : {dt*1e15:.0f} fs")
print(f"N           : {N}")

In [ ]:
# --------------------------------------------------------------------------
# Frequency axis and FFT
# --------------------------------------------------------------------------

# np.fft.fftfreq returns frequencies in cycles/sample  →  multiply by 1/dt
freq  = np.fft.fftfreq(N, d=dt)   # Hz
freq  = np.fft.fftshift(freq)     # centre DC at index N//2

# FFT
E_f   = np.fft.fft(E_t)
E_f   = np.fft.fftshift(E_f)     # match frequency axis
S     = np.abs(E_f)**2            # power spectral density

print(f"Frequency resolution : {freq[1]-freq[0]:.3e} Hz")
print(f"Max frequency        : {freq[-1]/1e12:.1f} THz")

In [ ]:
# --------------------------------------------------------------------------
# 🔬 Physics Insight — time–bandwidth product
# --------------------------------------------------------------------------

# FWHM in time: Δt_FWHM = 2√(2 ln 2) · τ
Δt_FWHM = 2 * np.sqrt(2 * np.log(2)) * τ

# Find spectral FWHM numerically
S_norm  = S / S.max()
half    = S_norm >= 0.5
Δν_FWHM = (freq[half].max() - freq[half].min())

TBP = Δt_FWHM * Δν_FWHM

print(f"Δt FWHM  : {Δt_FWHM*1e15:.1f} fs")
print(f"Δν FWHM  : {Δν_FWHM/1e12:.2f} THz")
print(f"TBP      : {TBP:.4f}")
print(f"Expected : {2*np.log(2)/np.pi:.4f}  (transform limit for Gaussian)")

In [ ]:
# ⚡ Try It — chirped pulse spectrum
#
# Add a quadratic spectral phase (GDD) to E_f:
#   φ(ω) = ½ · GDD · (ω - ω₀)²   where ω = 2π·freq
#
# GDD = 500 fs²
# IFFT back to time domain. How does the pulse duration change?
# Does the spectrum change?

# YOUR CODE HERE

# Solution (hidden)
# ---
# GDD    = 500e-30   # 500 fs²
# ω      = 2 * np.pi * freq
# ω0_val = 2 * np.pi * c / λ0
# phase  = 0.5 * GDD * (ω - ω0_val)**2
# E_f_chirped = np.fft.ifftshift(E_f) * np.exp(1j * np.fft.ifftshift(phase))
# E_t_chirped = np.fft.ifft(E_f_chirped)
# I_chirped   = np.abs(E_t_chirped)**2
# # Spectrum unchanged; pulse broadened by sqrt(1 + (GDD/τ²)²)

---
## Block 7 — Practical tips & I/O

Quick reference for the most useful utility functions.

In [ ]:
# --------------------------------------------------------------------------
# np.where, np.clip, NaN handling
# --------------------------------------------------------------------------

# np.where: element-wise conditional
signal = np.array([0.2, -0.1, 0.5, np.nan, 0.8])

# Replace values below threshold with 0
clean = np.where(signal > 0, signal, 0.0)
print(f"np.where : {clean}")

# np.clip: clamp to [low, high]
clipped = np.clip(signal, 0.0, 0.6)
print(f"np.clip  : {clipped}")

# NaN-safe aggregations
print(f"np.sum       : {np.sum(signal)}")         # nan propagates
print(f"np.nansum    : {np.nansum(signal)}")       # ignores NaN
print(f"np.nanmean   : {np.nanmean(signal):.3f}") # ignores NaN

In [ ]:
# --------------------------------------------------------------------------
# Array I/O
# --------------------------------------------------------------------------

import tempfile, os

# Save / load in binary format (.npy) — fast, lossless
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "pulse.npy")
    np.save(path, E_t)
    E_t_loaded = np.load(path)
    print(f"npy round-trip OK: {np.allclose(E_t, E_t_loaded)}")

    # Save multiple arrays in a single archive (.npz)
    path_z = os.path.join(tmp, "pulse_data.npz")
    np.savez(path_z, t=t, E_t=E_t, freq=freq, E_f=E_f)
    data = np.load(path_z)
    print(f"npz keys: {list(data.keys())}")

    # Text I/O (for sharing with other tools)
    path_txt = os.path.join(tmp, "intensity.txt")
    np.savetxt(path_txt, np.column_stack([t * 1e15, np.abs(E_t)**2]),
               header="time_fs  intensity", fmt="%.6e")
    data_txt = np.loadtxt(path_txt)
    print(f"txt shape: {data_txt.shape}")

💡 **Pro Tip — memory layout: C vs Fortran order**

By default NumPy arrays are **C-contiguous** (row-major): element `a[i, j]` is adjacent in memory to `a[i, j+1]`.  
Operations along the last axis are fastest.

Fortran order (column-major) is used by LAPACK and some legacy Fortran codes.  
We will revisit this in S09 when we look at performance and FFI.

```python
a = np.ones((1000, 1000))
# Fast: iterate along last axis (columns)
s = a.sum(axis=1)   # sums each row → O(N) cache hits
```

---
## Extending `shared/`

Before the exercises, migrate the new functions into the shared modules.  
Open a terminal in your course repo and add the following.

In [ ]:
# Functions to add to shared/optics.py

def gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray:
    """
    Radial intensity profile of a Gaussian beam.

    Parameters
    ----------
    r : np.ndarray
        Radial coordinate array (m).
    w : float
        Beam radius 1/e² (m).
    P : float
        Total power (W).

    Returns
    -------
    np.ndarray
        Intensity profile I(r) in W/m².
    """
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


def gaussian_field_2d(
    x: np.ndarray,
    y: np.ndarray,
    w: float,
    E0: float = 1.0,
) -> np.ndarray:
    """
    2D Gaussian electric field amplitude via broadcasting.

    Parameters
    ----------
    x, y : np.ndarray
        1D coordinate arrays (m). Output shape will be (len(y), len(x)).
    w : float
        Beam radius 1/e field (m).
    E0 : float
        Peak field amplitude.

    Returns
    -------
    np.ndarray
        Complex field array of shape (len(y), len(x)).
    """
    X = x[np.newaxis, :]   # (1, Nx)
    Y = y[:, np.newaxis]   # (Ny, 1)
    return E0 * np.exp(-(X**2 + Y**2) / w**2)


# Quick self-test
r_test = np.array([0.0, 500e-6])
I_test = gaussian_intensity(r_test, w=500e-6, P=1.0)
assert np.isclose(I_test[1] / I_test[0], np.exp(-2)), "gaussian_intensity failed"
print("gaussian_intensity ✓")

x_test = np.linspace(-1e-3, 1e-3, 32)
E_test = gaussian_field_2d(x_test, x_test, w=500e-6)
assert E_test.shape == (32, 32), "gaussian_field_2d shape failed"
print("gaussian_field_2d  ✓")

In [ ]:
# Functions to add to shared/pulses.py

def freq_axis(t: np.ndarray) -> np.ndarray:
    """
    Compute the FFT frequency axis for a uniformly sampled time array.

    Parameters
    ----------
    t : np.ndarray
        1D time array (s), uniformly spaced.

    Returns
    -------
    np.ndarray
        Frequency axis (Hz), centred (DC at index N//2).
    """
    N  = len(t)
    dt = t[1] - t[0]
    return np.fft.fftshift(np.fft.fftfreq(N, d=dt))


def pulse_spectrum(
    t: np.ndarray,
    E_t: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the power spectrum of a pulse field via FFT.

    Parameters
    ----------
    t : np.ndarray
        Time array (s).
    E_t : np.ndarray
        Complex or real electric field envelope.

    Returns
    -------
    freq : np.ndarray
        Frequency axis (Hz), centred.
    S : np.ndarray
        Power spectral density (a.u.), centred.
    """
    freq = freq_axis(t)
    E_f  = np.fft.fftshift(np.fft.fft(E_t))
    S    = np.abs(E_f)**2
    return freq, S


# Quick self-test
t_test  = np.linspace(-1e-12, 1e-12, 512)
E_test  = np.exp(-t_test**2 / (2 * (100e-15)**2))
freq_t, S_t = pulse_spectrum(t_test, E_test)
assert S_t.shape == t_test.shape
assert freq_t[len(freq_t)//2] == 0.0 or abs(freq_t[len(freq_t)//2]) < 1e9
print("freq_axis     ✓")
print("pulse_spectrum ✓")

---
## Summary

| Concept | Key functions |
|---------|---------------|
| Array creation | `np.array`, `np.zeros`, `np.ones`, `np.linspace`, `np.arange`, `np.eye` |
| Inspection | `.shape`, `.ndim`, `.size`, `.dtype`, `.itemsize`, `.nbytes` |
| Slicing | `a[i:j]`, `a[i:j, k:l]`, `a[mask]`, `a[idx_array]` |
| Ufuncs | `np.sin`, `np.exp`, `np.abs`, `np.angle`, `np.sqrt` |
| Aggregation | `.sum(axis=)`, `.max()`, `np.cumsum`, `np.diff`, `np.argmax` |
| Broadcasting | `np.newaxis`, `None`, shape alignment rules |
| Linear algebra | `@`, `np.linalg.solve`, `np.linalg.eig`, `np.linalg.inv` |
| FFT | `np.fft.fft`, `np.fft.fftfreq`, `np.fft.fftshift` |
| Utilities | `np.where`, `np.clip`, `np.isnan`, `np.nansum`, `np.allclose` |
| I/O | `np.save/load`, `np.savez`, `np.savetxt/loadtxt` |

**`shared/` additions:**
- `shared/optics.py` ← `gaussian_intensity`, `gaussian_field_2d`
- `shared/pulses.py` ← `freq_axis`, `pulse_spectrum`

**Next: S05 — Matplotlib**